# Figures & tables for the paper

Curated, paper-ready figures and LaTeX-exportable tables, pulled from the same
output trees every other analysis notebook in this repo reads
(`outputs_classifier`, `outputs_classifier_colon`, `outputs_probing_colon`,
`outputs_deepspot_colon`). Where `plots_for_paper.ipynb` covers the probing
comparisons and `compare_classifier_results.ipynb` / `saved_cells_deep_dive.ipynb`
/ `analyse_probing_results.ipynb` cover exploratory dashboards, this notebook is
the final, specific set of comparisons requested for the paper -- one config
cell drives everything, and every table/plot below **reports what's missing**
(0 splits found, some splits found, wrong folder name) rather than silently
producing a wrong or incomplete number.

**`same_wsi_split` is excluded everywhere** (it's a random pooled 70/15/15
sanity-check split, not a LOSO fold -- see `src/python/classifier_training/README.md`).

## Sections

1. Performance across models and datasets (central vs. CLS-native vs. CLS-100resized)
2. Performance vs. other methods (cross-cancer): advantage over the Dummy(PCA) baseline
3. UNI2 colon: performance across resize field-of-view
4. Neighbor-cell-type baseline at multiple radii (56/112/224/448px)
5. Probing: `cell_count`/`cell_density`, central vs. CLS, 100-resized vs. native
6. Probing: Phikon-v2 `area` probe, masked vs. unmasked (central tokens)
7. Classification: Phikon-v2 colon classification, masked vs. unmasked (central tokens)
8. Classification: UNI2 cross-cancer, central vs. default tokens
9. Masking-drift vs. biological difference: top/bottom-drift cell distributions
10. RNA-context divergence vs. masking disagreement: distributions
11. Multicell UNI2 (colon): performance vs. forward passes / reduction factor
12. DeepSpot colon: UNI2 vs. UNI2 (resized), fold-change vs. 30 neighbours

## Before you run this

None of `outputs_classifier`, `outputs_classifier_colon`, `outputs_probing_colon`,
`outputs_deepspot_colon` exist on this machine as of writing -- this notebook was
authored against the documented output layout (see each pipeline's README) and the
model-name conventions already established in `plots_for_paper.ipynb` /
`batch_effects_across_models.ipynb` / `compare_classifier_results.ipynb`, **not
run end-to-end against real data**. Place (or mount) those four folders as
siblings of `notebooks/` (i.e. under the repo root), then run top to bottom.
The single CONFIG cell below (`MODEL_LABELS` etc.) is the one place to fix a
wrong folder-name guess -- every loader warns rather than fails if a configured
name isn't found on disk.


## Imports & setup

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial import cKDTree
from scipy.stats import mannwhitneyu
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


def find_repo_root(start: Path) -> Path:
    # Jupyter's cwd is wherever the server was launched from (usually notebooks/),
    # not necessarily the repo root -- same pattern as saved_cells_deep_dive.ipynb.
    for d in [start, *start.parents]:
        if (d / "src" / "python" / "code_configs").is_dir():
            return d
    raise RuntimeError(f"could not locate repo root above {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT = {REPO_ROOT}")


def load_dotenv(path: Path) -> None:
    if not path.exists():
        print(f"[no .env at {path} -- env-var-backed paths below will raise until you create one]")
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())


load_dotenv(REPO_ROOT / ".env")

from src.python.code_configs import paths as P  # noqa: E402
from src.python.code_configs.mappings import mapping_factory  # noqa: E402

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "font.size": 9,
    "legend.fontsize": 8,
})


## CONFIG -- everything below is parameterized by this cell

`MODEL_LABELS`'s keys are the exact `model_name` path segment (the top-level
folder name under each results root). **Edit this dict first** if your actual
folder names differ from the guesses below -- every table/plot only ever
references these same keys, and every loader warns (rather than silently
producing a wrong number) about a configured name it can't find on disk, or a
name found on disk that isn't in this dict.

Conventions reused as-is from `plots_for_paper.ipynb` / `batch_effects_across_models.ipynb`
/ `compare_classifier_results.ipynb` (already validated there): central token =
mean of the four native corner tokens; `correction_name="raw"` throughout (per
your instruction); colon correction is *not* forced to `"Base"` here even
though `plots_for_paper.ipynb` used that for probing -- you said to always use
`"raw"`, so that's the one default below, with the usual "0 rows, falling back"
warning if a root only has `"Base"`.


In [ ]:
CLASSIFIER_ROOTS = {
    "cross_cancer": Path("../outputs_classifier"),
    "colon":        Path("../outputs_classifier_colon"),
}
PROBING_ROOTS = [Path("../outputs_probing_colon")]
DEEPSPOT_COLON_ROOT = Path("../outputs_deepspot_colon")

EXPECTED_SPLITS = {"cross_cancer": 14, "colon": 8}  # configs/train.yaml / train_colon.yaml split counts
EXPECTED_SPLITS_PROBING = 8                          # configs/probing_colon.yaml split count
EXPECTED_SPLITS_DEEPSPOT_COLON = 4                   # configs/deepspot_train_colon.yaml split count

# Per your instruction: always use the "raw" correction_name (not Base/rpca/scanorama).
CORRECTION_NAME = "raw"
MAPPING = None    # e.g. "simplified_broad" -- force one value if a root mixes several; None = auto-detect
MATCHING = None   # e.g. "all_cells" -- ditto

# ── tokens ("embeddings" path segment) ──────────────────────────────────────
def embeddings_segment(keys: list[str]) -> str:
    # Mirrors experiment.py's folder-naming: data.embeddings_datasets, sorted and
    # "_"-joined (or "default" if empty -> average every embeddings_* key in the h5).
    return "_".join(sorted(keys)) if keys else "default"


TOKEN_CLS = "cls"
TOKEN_CENTRAL_KEYS = ["bottom_left", "bottom_right", "top_left", "top_right"]
TOKEN_CENTRAL = embeddings_segment(TOKEN_CENTRAL_KEYS)   # "bottom_left_bottom_right_top_left_top_right"
TOKEN_DEFAULT = embeddings_segment([])                    # "default"

TOKEN_LABELS = {TOKEN_CLS: "CLS", TOKEN_CENTRAL: "central (4-token avg)", TOKEN_DEFAULT: "default (all tokens avg)"}

# ── models: model_name (folder) -> display label ────────────────────────────
MODEL_LABELS: dict[str, str] = {
    # native (no resize) extraction -- central tokens + cls in the same h5 file
    "UNI2_specific_tokens_folder": "UNI2",
    "HOptimus1":                   "H-Optimus-1",
    "PhikonV2_448":                "Phikon-v2",
    "virchow_v2":                  "Virchow2",
    # 100px-crop-resized-to-224 baseline (rho_CLS^resize)
    "UNI2_100_resized":            "UNI2",
    "HOptimus1_100_resized":       "H-Optimus-1",
    "PhikonV2_100_resized":        "Phikon-v2",
    "virchow_v2_100_resized":      "Virchow2",
    # masked (central 3x3 tokens masked at inference)
    "PhikonV2_448_masked":         "Phikon-v2 (masked)",
    # UNI2 colon field-of-view / resize sweep (all central tokens)
    "UNI2_1344_resized":           "UNI2 (1344px)",
    "UNI2_896_resized":            "UNI2 (896px)",
    # UNI2_specific_tokens_folder doubles as this sweep's 224px (native) point.
    # deepspot colon: UNI2 vs. a smaller resize
    "UNI2_56_resized":             "UNI2 (56px)",
    # other (non-foundation-model) baselines, cross-cancer only
    "Dummy":                       "Dummy (PCA)",
    "CONCH":                       "CONCH",
    "CTransPath":                  "CTransPath",
    "CellViT_SAM":                 "CellViT_SAM",
    # multicell UNI2 (colon), one entry per central-window size
    "UNI2_448_112_multicell":      "UNI2 multicell (112px window)",
    "UNI2_448_224_multicell":      "UNI2 multicell (224px window)",
    "UNI2_448_448_multicell":      "UNI2 multicell (448px window)",
}

FOUNDATION_MODELS = ["UNI2_specific_tokens_folder", "HOptimus1", "PhikonV2_448", "virchow_v2"]
OTHER_METHODS = ["CONCH", "CTransPath", "CellViT_SAM"]
DUMMY_BASELINE = "Dummy"

# family (native model_name) -> its 100px-resized counterpart model_name.
RESOLUTION_FAMILIES_BY_NATIVE: dict[str, str] = {
    "UNI2_specific_tokens_folder": "UNI2_100_resized",
    "HOptimus1":                   "HOptimus1_100_resized",
    "PhikonV2_448":                "PhikonV2_100_resized",
    "virchow_v2":                  "virchow_v2_100_resized",
}

MASK_COMPARISON = {"unmasked": "PhikonV2_448", "masked": "PhikonV2_448_masked"}

# UNI2 colon field-of-view / resize sweep -- size_side (native crop px) -> model_name.
# NOTE: 100px is used as the "smallest" baseline for now per your instruction
# ("I dont have it yet, for the moment use 100 as the minimum but it will be
# changed later") -- swap/add an entry here once a dedicated smaller run exists.
UNI2_RESIZE_SWEEP: dict[int, str] = {
    1344: "UNI2_1344_resized",
    896:  "UNI2_896_resized",
    224:  "UNI2_specific_tokens_folder",
    100:  "UNI2_100_resized",
}
UNI2_RESIZE_SWEEP_TOKEN = TOKEN_CENTRAL  # confirmed: this whole sweep is read via the central token

# DeepSpot colon: UNI2 vs. a smaller resize, across n_neighbors 30/15/0.
DEEPSPOT_MODELS = {"UNI2": "UNI2_specific_tokens_folder", "UNI2 resized": "UNI2_56_resized"}
DEEPSPOT_NEIGHBOURS = [30, 15, 0]

# Multicell UNI2 (colon) -- one entry per central-window size. Forward-pass counts
# are filled in *manually* here (no JSON, per your instruction) -- fill in the
# Nones below before running section 11. reduction_factor = FORWARD_PASSES[CLS_BASELINE_MODEL] / FORWARD_PASSES[model].
MULTICELL_MODELS = ["UNI2_448_112_multicell", "UNI2_448_224_multicell", "UNI2_448_448_multicell"]
MULTICELL_TOKEN = "cell_token"   # extract_embeddings_multicell's single per-cell token (no CLS/corner split)
CLS_BASELINE_MODEL = "UNI2_specific_tokens_folder"  # dashed baseline: UNI2 colon classifier, CLS token

FORWARD_PASSES: dict[str, "int | None"] = {
    "UNI2_specific_tokens_folder": None,   # TODO: fill in the plain per-cell UNI2 forward-pass count
    "UNI2_448_112_multicell":      None,   # TODO
    "UNI2_448_224_multicell":      None,   # TODO
    "UNI2_448_448_multicell":      None,   # TODO
}

# ── probing: probes & context windows (configs/probing_colon.yaml) ──────────
CONTEXT_PROBES_OF_INTEREST = ["cell_count", "cell_density"]
CONTEXT_PX_LIST = [224, 448, 1344]
CELL_LEVEL_PROBE = "area"

METRICS = ["test_accuracy", "test_balanced_accuracy", "test_macro_f1", "test_weighted_f1"]
PRIMARY_METRIC = "test_macro_f1"

# ── figure / table export ────────────────────────────────────────────────────
SAVE_FIGURES = False
FIGURES_DIR = Path("figures_for_paper")
LATEX_DIR = Path("tables_for_paper")


## Shared helpers

Generic loaders + stats helpers reused by every section below. Classifier/probing loaders mirror `compare_classifier_results.ipynb` / `plots_for_paper.ipynb` exactly (same schema, same missing-data reporting philosophy) so results here are directly comparable to those notebooks.

In [ ]:
def show_table(df: pd.DataFrame, name: str, caption: str = "", float_format: str = "%.4f") -> pd.DataFrame:
    """Display df and print/save its LaTeX form under LATEX_DIR/{name}.tex."""
    display(df)
    try:
        latex = df.to_latex(index=False, float_format=float_format,
                             caption=caption or name.replace("_", " "), label=f"tab:{name}")
    except Exception as e:
        print(f"[to_latex failed: {e}]")
        return df
    print(latex)
    if SAVE_FIGURES:
        LATEX_DIR.mkdir(parents=True, exist_ok=True)
        (LATEX_DIR / f"{name}.tex").write_text(latex)
        print(f"[saved] {LATEX_DIR / f'{name}.tex'}")
    return df


def save_fig(fig, name: str) -> None:
    if not SAVE_FIGURES:
        return
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    for ext in ("png", "pdf"):
        fig.savefig(FIGURES_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")
    print(f"[save_fig] wrote {FIGURES_DIR / name}.{{png,pdf}}")


def fmt_mean_std(mean, std, n, expected=None, digits=4) -> str:
    if n == 0 or mean is None or (isinstance(mean, float) and np.isnan(mean)):
        return "missing"
    s = f"{mean:.{digits}f} \u00b1 {std:.{digits}f}"
    if expected is not None and n < expected:
        s += f" (n={n}/{expected})"
    return s


In [ ]:
CLASSIFIER_SCHEMA = ("model_name", "correction_name", "mapping", "matching", "embeddings", "split_label")


def _flatten(prefix: str, d: dict) -> dict:
    return {f"{prefix}_{k}": v for k, v in d.items()}


def load_classifier_results(root: Path, dataset: str) -> pd.DataFrame:
    """One row per LOSO run under root (results_summary.yaml), tagged with `dataset`."""
    rows: list[dict] = []
    for summary_path in sorted(root.rglob("results_summary.yaml")):
        parts = summary_path.relative_to(root).parts[:-1]
        if len(parts) != len(CLASSIFIER_SCHEMA):
            print(f"[skip] unexpected path depth ({len(parts)} != {len(CLASSIFIER_SCHEMA)}): {summary_path}")
            continue
        meta = dict(zip(CLASSIFIER_SCHEMA, parts))
        with open(summary_path) as f:
            summary = yaml.safe_load(f)

        split_label = meta["split_label"]
        is_same_wsi = split_label == "same_wsi_split"
        split_idx = None if is_same_wsi else int(split_label.removeprefix("split_"))

        rows.append({
            "dataset": dataset, **meta, "split_idx": split_idx, "is_same_wsi": is_same_wsi,
            **_flatten("cfg", summary.get("best_config", {})),
            **_flatten("val", summary.get("best_val_metrics", {})),
            **_flatten("test", summary.get("best_test_metrics", {})),
        })
    return pd.DataFrame(rows)


def load_all_classifier_results(roots: dict[str, Path]) -> pd.DataFrame:
    frames = []
    for dataset, root in roots.items():
        if not root.exists():
            print(f"[missing root] {dataset}: {root.resolve()}")
            continue
        df = load_classifier_results(root, dataset)
        if df.empty:
            print(f"[empty] no results_summary.yaml found under {root.resolve()}")
            continue
        print(f"[loaded] {dataset}: {len(df)} run rows, model_name folders found: {sorted(df['model_name'].unique())}")
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    return out[~out["is_same_wsi"]].copy()  # same_wsi_split excluded everywhere in this notebook


def filter_correction(df: pd.DataFrame, correction_name: str = CORRECTION_NAME) -> pd.DataFrame:
    if df.empty:
        return df
    keep = []
    for dataset, sub in df.groupby("dataset"):
        avail = sorted(sub["correction_name"].unique())
        filtered = sub[sub["correction_name"] == correction_name]
        if filtered.empty:
            print(f"[warning] dataset={dataset!r}: correction_name={correction_name!r} matches 0 rows -- "
                  f"available: {avail}. Keeping all rows for this dataset instead.")
            filtered = sub
        keep.append(filtered)
    return pd.concat(keep, ignore_index=True)


def filter_one_value(df: pd.DataFrame, col: str, forced) -> pd.DataFrame:
    if df.empty or forced is None:
        if not df.empty and col in df:
            values = sorted(df[col].dropna().unique().tolist())
            if len(values) > 1:
                print(f"[warning] multiple {col!r} values found {values} -- "
                      f"set {col.upper()} above to filter to one; keeping all of them for now")
        return df
    filtered = df[df[col] == forced]
    if filtered.empty:
        print(f"[warning] {col}={forced!r} matches 0 rows. Keeping all rows instead.")
        return df
    return filtered


CLASSIFIER_DF = load_all_classifier_results(CLASSIFIER_ROOTS)
if not CLASSIFIER_DF.empty:
    CLASSIFIER_DF = filter_correction(CLASSIFIER_DF)
    CLASSIFIER_DF = filter_one_value(CLASSIFIER_DF, "mapping", MAPPING)
    CLASSIFIER_DF = filter_one_value(CLASSIFIER_DF, "matching", MATCHING)
    print(f"\nFinal CLASSIFIER_DF: {len(CLASSIFIER_DF)} rows.")
    _unknown = sorted(set(CLASSIFIER_DF["model_name"].unique()) - set(MODEL_LABELS))
    if _unknown:
        print(f"[warning] model_name folders found on disk but missing from MODEL_LABELS: {_unknown} -- add them there.")
else:
    print("[warning] CLASSIFIER_DF is empty -- every classifier-based section below will report everything missing.")


In [ ]:
PROBE_SCHEMA = ("model_name", "correction_name", "mapping", "matching", "embeddings")


def _split_idx(split_label: str):
    return None if split_label == "same_wsi_split" else int(split_label.removeprefix("split_"))


def load_probe_summaries(root: Path) -> pd.DataFrame:
    rows: list[dict] = []
    for summary_path in sorted(root.rglob("probes_summary.yaml")):
        parts = summary_path.relative_to(root).parts[:-1]
        if len(parts) != len(PROBE_SCHEMA) + 1:  # + split_label
            print(f"[skip] unexpected path depth ({len(parts)} != {len(PROBE_SCHEMA) + 1}): {summary_path}")
            continue
        meta = dict(zip(PROBE_SCHEMA, parts[:-1]))
        split_label = parts[-1]
        with open(summary_path) as f:
            probes = yaml.safe_load(f) or {}
        for probe_name, entry in probes.items():
            rows.append({
                **meta, "split_label": split_label, "split_idx": _split_idx(split_label),
                "is_same_wsi": split_label == "same_wsi_split",
                "probe_dir": summary_path.parent / probe_name.replace("@", "_"),
                "probe": probe_name, **(entry or {}),
            })
    return pd.DataFrame(rows)


def load_probe_details(df: pd.DataFrame) -> pd.DataFrame:
    extra_rows = []
    for _, row in df.iterrows():
        extra: dict = {}
        if not row.get("skipped", True):
            path = Path(row["probe_dir"]) / "results_summary.yaml"
            if path.exists():
                with open(path) as f:
                    summary = yaml.safe_load(f)
                val_metrics = dict(summary.get("best_val_metrics") or {})
                test_metrics = dict(summary.get("best_test_metrics") or {})
                val_metrics.pop("selection_metric", None)
                test_metrics.pop("selection_metric", None)
                extra = {
                    "n_train": summary.get("n_train"), "n_val": summary.get("n_val"), "n_test": summary.get("n_test"),
                    **_flatten("val", val_metrics), **_flatten("test", test_metrics),
                }
        extra_rows.append(extra)
    extra_df = pd.DataFrame(extra_rows, index=df.index)
    return pd.concat([df, extra_df], axis=1)


def load_all_probe_results(roots: list[Path]) -> pd.DataFrame:
    frames = []
    for root in roots:
        if not root.exists():
            print(f"[missing root] {root.resolve()}")
            continue
        df = load_probe_summaries(root)
        if df.empty:
            print(f"[empty] no probes_summary.yaml found under {root.resolve()}")
            continue
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    out = load_probe_details(out)
    return out[~out["is_same_wsi"]].copy()


PROBE_DF = load_all_probe_results(PROBING_ROOTS)
if not PROBE_DF.empty:
    PROBE_DF = filter_correction(PROBE_DF.assign(dataset="colon")).drop(columns=["dataset"])
    PROBE_DF = filter_one_value(PROBE_DF, "mapping", MAPPING)
    PROBE_DF = filter_one_value(PROBE_DF, "matching", MATCHING)
    print(f"Loaded {len(PROBE_DF)} (run x probe) rows. model_name folders found: {sorted(PROBE_DF['model_name'].unique())}")
    print(f"embeddings segments found: {sorted(PROBE_DF['embeddings'].unique())}")
    print(f"probes found: {sorted(PROBE_DF['probe'].unique())}")
    _unknown = sorted(set(PROBE_DF["model_name"].unique()) - set(MODEL_LABELS))
    if _unknown:
        print(f"[warning] model_name folders found on disk but missing from MODEL_LABELS: {_unknown} -- add them there.")
else:
    print("[warning] PROBE_DF is empty -- every probing-based section below will report everything missing.")


In [ ]:
DEEPSPOT_SCHEMA = ("model_name", "correction_name", "matching", "embeddings", "num_genes", "num_neighbours", "split_label")


def load_deepspot_results(root: Path) -> pd.DataFrame:
    """See src/python/deepspot_training/README.md's Output structure. Grabs every
    *_pearson-ish key found in results_summary.yaml rather than guessing one exact
    name (the README doesn't spell it out) -- check the printed columns and set
    DEEPSPOT_METRIC in section 12 if the auto-picked one isn't right."""
    rows: list[dict] = []
    for summary_path in sorted(root.rglob("results_summary.yaml")):
        parts = summary_path.relative_to(root).parts[:-1]
        if len(parts) != len(DEEPSPOT_SCHEMA):
            print(f"[skip] unexpected path depth ({len(parts)} != {len(DEEPSPOT_SCHEMA)}): {summary_path}")
            continue
        meta = dict(zip(DEEPSPOT_SCHEMA, parts))
        with open(summary_path) as f:
            summary = yaml.safe_load(f)
        split_label = meta["split_label"]
        is_same_wsi = split_label == "same_wsi_split"
        split_idx = None if is_same_wsi else int(split_label.removeprefix("split_"))
        pearson_like = {k: v for k, v in (summary or {}).items()
                        if isinstance(k, str) and "pearson" in k.lower() and not isinstance(v, (dict, list))}
        num_neighbours = meta["num_neighbours"]
        try:
            num_neighbours = int(num_neighbours)
        except ValueError:
            pass
        rows.append({**meta, "num_neighbours": num_neighbours, "split_idx": split_idx,
                     "is_same_wsi": is_same_wsi, **pearson_like})
    return pd.DataFrame(rows)


DEEPSPOT_DF = pd.DataFrame()
if DEEPSPOT_COLON_ROOT.exists():
    DEEPSPOT_DF = load_deepspot_results(DEEPSPOT_COLON_ROOT)
    if DEEPSPOT_DF.empty:
        print(f"[empty] no results_summary.yaml found under {DEEPSPOT_COLON_ROOT.resolve()}")
    else:
        DEEPSPOT_DF = DEEPSPOT_DF[~DEEPSPOT_DF["is_same_wsi"]].copy()
        DEEPSPOT_DF = filter_correction(DEEPSPOT_DF.assign(dataset="colon")).drop(columns=["dataset"])
        _pearson_cols = [c for c in DEEPSPOT_DF.columns if "pearson" in c.lower()]
        print(f"Loaded {len(DEEPSPOT_DF)} rows. model_name folders found: {sorted(DEEPSPOT_DF['model_name'].unique())}")
        print(f"num_neighbours values found: {sorted(DEEPSPOT_DF['num_neighbours'].unique(), key=str)}")
        print(f"pearson-like columns found: {_pearson_cols}")
else:
    print(f"[missing root] {DEEPSPOT_COLON_ROOT.resolve()}")


### Stats helpers

`mean_std_n` / `paired_diff` operate on `CLASSIFIER_DF`; the `*_probe` variants operate on `PROBE_DF` (extra `probe` filter, `skipped == False`, metric defaults to `test_selection_metric` -- R² for cell_count/cell_density/area). Every diff is a *paired* per-split difference (matched on `split_idx`), then mean/std across the matched splits -- not a difference of means -- so it only ever uses splits present on both sides of the comparison.

In [ ]:
def mean_std_n(df: pd.DataFrame, dataset: str, model: str, embeddings: str,
               metric: str = PRIMARY_METRIC) -> tuple[float, float, int]:
    if df.empty:
        return float("nan"), float("nan"), 0
    sub = df[(df["dataset"] == dataset) & (df["model_name"] == model) & (df["embeddings"] == embeddings)]
    vals = sub[metric].dropna() if metric in sub else pd.Series(dtype=float)
    n = len(vals)
    if n == 0:
        return float("nan"), float("nan"), 0
    return float(vals.mean()), float(vals.std()) if n > 1 else 0.0, n


def paired_diff(df: pd.DataFrame, dataset: str, model_a: str, emb_a: str, model_b: str, emb_b: str,
                metric: str = PRIMARY_METRIC) -> tuple[float, float, int]:
    if df.empty or model_a is None or model_b is None:
        return float("nan"), float("nan"), 0
    a = df[(df["dataset"] == dataset) & (df["model_name"] == model_a) & (df["embeddings"] == emb_a)][["split_idx", metric]]
    b = df[(df["dataset"] == dataset) & (df["model_name"] == model_b) & (df["embeddings"] == emb_b)][["split_idx", metric]]
    merged = a.merge(b, on="split_idx", suffixes=("_a", "_b"))
    diffs = (merged[f"{metric}_a"] - merged[f"{metric}_b"]).dropna()
    n = len(diffs)
    if n == 0:
        return float("nan"), float("nan"), 0
    return float(diffs.mean()), float(diffs.std()) if n > 1 else 0.0, n


def mean_std_metric_probe(df: pd.DataFrame, model: str, embeddings: str, probe: str,
                           metric: str = "test_selection_metric") -> tuple[float, float, int]:
    if df.empty or model is None:
        return float("nan"), float("nan"), 0
    sub = df[(df["model_name"] == model) & (df["embeddings"] == embeddings)
             & (df["probe"] == probe) & (df["skipped"] == False)]  # noqa: E712
    vals = sub[metric].dropna() if metric in sub else pd.Series(dtype=float)
    n = len(vals)
    if n == 0:
        return float("nan"), float("nan"), 0
    return float(vals.mean()), float(vals.std()) if n > 1 else 0.0, n


def paired_diff_probe(df: pd.DataFrame, model_a: str, emb_a: str, model_b: str, emb_b: str, probe: str,
                       metric: str = "test_selection_metric") -> tuple[float, float, int]:
    if df.empty or model_a is None or model_b is None:
        return float("nan"), float("nan"), 0
    a = df[(df["model_name"] == model_a) & (df["embeddings"] == emb_a) & (df["probe"] == probe)
           & (df["skipped"] == False)][["split_idx", metric]]  # noqa: E712
    b = df[(df["model_name"] == model_b) & (df["embeddings"] == emb_b) & (df["probe"] == probe)
           & (df["skipped"] == False)][["split_idx", metric]]  # noqa: E712
    merged = a.merge(b, on="split_idx", suffixes=("_a", "_b"))
    diffs = (merged[f"{metric}_a"] - merged[f"{metric}_b"]).dropna()
    n = len(diffs)
    if n == 0:
        return float("nan"), float("nan"), 0
    return float(diffs.mean()), float(diffs.std()) if n > 1 else 0.0, n


def probe_key(name: str, context_px) -> str:
    return f"{name}@{context_px}px" if context_px is not None else name


## 1. Performance across models and datasets

Cross-cancer and colon, for all 4 foundation models: central tokens (normal/native
extraction) vs. CLS (normal/native extraction) vs. CLS (100px-crop-resized
extraction). Raw table first, then a table of the two paired differences
(central - CLS-native, and CLS-native - CLS-100resized), mean \u00b1 std across LOSO
splits (paired by `split_idx`, so a split missing on one side just isn't counted --
see `n` columns).

In [ ]:
def build_model_comparison_tables(dataset: str, models: list[str] = FOUNDATION_MODELS):
    expected = EXPECTED_SPLITS.get(dataset)
    raw_rows, diff_rows = [], []
    for model in models:
        label = MODEL_LABELS.get(model, model)
        resized_model = RESOLUTION_FAMILIES_BY_NATIVE.get(model)

        m_central, s_central, n_central = mean_std_n(CLASSIFIER_DF, dataset, model, TOKEN_CENTRAL)
        m_cls, s_cls, n_cls = mean_std_n(CLASSIFIER_DF, dataset, model, TOKEN_CLS)
        m_cls100, s_cls100, n_cls100 = mean_std_n(CLASSIFIER_DF, dataset, resized_model, TOKEN_CLS)

        raw_rows.append({
            "model": label,
            "central_mean": m_central, "central_std": s_central, "central_n": n_central,
            "cls_native_mean": m_cls, "cls_native_std": s_cls, "cls_native_n": n_cls,
            "cls_100resized_mean": m_cls100, "cls_100resized_std": s_cls100, "cls_100resized_n": n_cls100,
        })

        d1_mean, d1_std, d1_n = paired_diff(CLASSIFIER_DF, dataset, model, TOKEN_CENTRAL, model, TOKEN_CLS)
        d2_mean, d2_std, d2_n = paired_diff(CLASSIFIER_DF, dataset, model, TOKEN_CLS, resized_model, TOKEN_CLS)
        diff_rows.append({
            "model": label,
            "central_minus_cls_native_mean": d1_mean, "central_minus_cls_native_std": d1_std, "central_minus_cls_native_n": d1_n,
            "cls_native_minus_cls_100resized_mean": d2_mean, "cls_native_minus_cls_100resized_std": d2_std,
            "cls_native_minus_cls_100resized_n": d2_n,
        })

    raw_df, diff_df = pd.DataFrame(raw_rows), pd.DataFrame(diff_rows)
    for col in ("central_n", "cls_native_n", "cls_100resized_n"):
        n_missing = (raw_df[col] == 0).sum()
        if n_missing:
            print(f"[{dataset}] {n_missing}/{len(raw_df)} model(s) have 0 splits for column {col!r} -- see table below.")
    print(f"[{dataset}] expected {expected} LOSO splits per condition.")
    return raw_df, diff_df


for _dataset in ("cross_cancer", "colon"):
    print(f"\n=== {_dataset}: raw macro-F1 (mean \u00b1 std, n splits) ===")
    _raw, _diff = build_model_comparison_tables(_dataset)
    show_table(_raw.round(4), f"model_comparison_raw_{_dataset}")
    print(f"\n=== {_dataset}: paired differences (macro-F1) ===")
    show_table(_diff.round(4), f"model_comparison_diff_{_dataset}")


## 2. Performance vs. other methods (cross-cancer)

Mean advantage (paired, per-split) of macro-F1 over the `Dummy` (PCA-over-raw-pixels)
baseline -- each of the 4 foundation models' central tokens, plus the other
non-foundation-model providers implemented in this repo (`CONCH`, `CTransPath`,
`CellViT_SAM`), each read via their own `default` embeddings segment (these
providers save a single named token each -- `center`/`pixels`/etc. -- so `default`
i.e. "average every embeddings_* key found" is equivalent to reading that one token).

In [ ]:
def build_other_methods_table(dataset: str = "cross_cancer"):
    rows = []
    for model in FOUNDATION_MODELS:
        mean_d, std_d, n_d = paired_diff(CLASSIFIER_DF, dataset, model, TOKEN_CENTRAL, DUMMY_BASELINE, TOKEN_DEFAULT)
        rows.append({"model": MODEL_LABELS.get(model, model), "token": TOKEN_LABELS[TOKEN_CENTRAL],
                     "advantage_over_dummy_mean": mean_d, "advantage_over_dummy_std": std_d, "n_splits": n_d})
    for model in OTHER_METHODS:
        mean_d, std_d, n_d = paired_diff(CLASSIFIER_DF, dataset, model, TOKEN_DEFAULT, DUMMY_BASELINE, TOKEN_DEFAULT)
        rows.append({"model": MODEL_LABELS.get(model, model), "token": TOKEN_LABELS[TOKEN_DEFAULT],
                     "advantage_over_dummy_mean": mean_d, "advantage_over_dummy_std": std_d, "n_splits": n_d})
    return pd.DataFrame(rows)


_other_methods_df = build_other_methods_table()
print(f"[cross_cancer] expected {EXPECTED_SPLITS['cross_cancer']} LOSO splits per comparison.")
show_table(_other_methods_df.round(4), "other_methods_advantage_over_dummy")


## 3. UNI2 colon: performance across resize field-of-view

Central-token macro-F1 across `size_side` \u2208 {1344, 896, 224, 100} (224 = the
native/no-resize `UNI2_specific_tokens_folder` extraction). Improvement is the
paired, per-split difference over the smallest available size (currently 100px --
a placeholder per your note; swap `UNI2_RESIZE_SWEEP` in CONFIG once a dedicated
smaller run lands, nothing else here needs to change).

In [ ]:
def build_resize_sweep_table(dataset: str = "colon"):
    sizes = sorted(UNI2_RESIZE_SWEEP)  # ascending, e.g. [100, 224, 896, 1344]
    baseline_size = sizes[0]
    baseline_model = UNI2_RESIZE_SWEEP[baseline_size]
    print(f"[baseline] smallest available size_side = {baseline_size}px ({baseline_model})")

    rows = []
    for size in sizes:
        model = UNI2_RESIZE_SWEEP[size]
        m, s, n = mean_std_n(CLASSIFIER_DF, dataset, model, UNI2_RESIZE_SWEEP_TOKEN)
        imp_m, imp_s, imp_n = paired_diff(CLASSIFIER_DF, dataset, model, UNI2_RESIZE_SWEEP_TOKEN,
                                           baseline_model, UNI2_RESIZE_SWEEP_TOKEN)
        rows.append({"size_side_px": size, "model_name": model, "macro_f1_mean": m, "macro_f1_std": s, "n_splits": n,
                     "improvement_over_smallest_mean": imp_m, "improvement_over_smallest_std": imp_s,
                     "improvement_over_smallest_n": imp_n})
    print(f"[colon] expected {EXPECTED_SPLITS['colon']} LOSO splits per size.")
    return pd.DataFrame(rows)


_resize_df = build_resize_sweep_table()
show_table(_resize_df.round(4), "uni2_colon_resize_sweep")

_plot_df = _resize_df[_resize_df["improvement_over_smallest_n"] > 0]
if not _plot_df.empty:
    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(_plot_df))
    ax.bar(x, _plot_df["improvement_over_smallest_mean"], yerr=_plot_df["improvement_over_smallest_std"], capsize=3,
           color="#4C72B0")
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{s}px" for s in _plot_df["size_side_px"]])
    ax.set_xlabel("size_side (native crop px, resized to model input)")
    ax.set_ylabel(f"improvement over {_resize_df['size_side_px'].min()}px baseline (macro-F1)")
    ax.set_title("UNI2 colon: central-token macro-F1 improvement over smallest resize")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    save_fig(fig, "uni2_colon_resize_sweep")
    plt.show()
else:
    print("[skip plot] no size has a paired improvement value yet -- see [missing]/0-n rows above.")


## 4. Neighbor-cell-type baseline at multiple radii (56/112/224/448px)

Extends `saved_cells_deep_dive.ipynb`'s \u00a73 (`evaluate_neighbor_baseline`,
originally 112/224px only) to 56/112/224/448px. For each `radius_px`: majority
**ground-truth** label among same-WSI neighbors within that radius (self excluded),
scored against `true_label` the same way the model itself is scored -- a purely
spatial diagnostic upper bound, not a real classifier. Needs `test_predictions.csv`
(`training.save_test_predictions: true`, colon only) and `patch_coordinates.h5`
(`XENIUM_PROCESSED_OUTPUT_ROOT`).

In [ ]:
from sklearn.metrics import f1_score, balanced_accuracy_score  # noqa: E402

NEIGHBOR_RADII_PX = [56, 112, 224, 448]

# (model_name, embeddings) pairs to evaluate -- add rows here for more conditions.
NEIGHBOR_TEST_PREDICTIONS = [
    {"model_name": "UNI2_specific_tokens_folder", "embeddings": TOKEN_CLS},
    {"model_name": "UNI2_specific_tokens_folder", "embeddings": TOKEN_CENTRAL},
]
NEIGHBOR_MAPPING = "simplified_broad"
NEIGHBOR_MATCHING = "all_cells"


def load_test_predictions(root: Path, model_name: str, embeddings: str,
                           correction_name: str = CORRECTION_NAME, mapping: str = NEIGHBOR_MAPPING,
                           matching: str = NEIGHBOR_MATCHING) -> pd.DataFrame:
    base = root / model_name / correction_name / mapping / matching / embeddings
    if not base.exists():
        print(f"[missing] {base}")
        return pd.DataFrame()
    frames = []
    for split_dir in sorted(base.glob("split_*")):
        csv_path = split_dir / "test_predictions.csv"
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        df["correct"] = df["correct"].astype(bool)
        df["split_idx"] = int(split_dir.name.removeprefix("split_"))
        df["model_name"], df["embeddings"] = model_name, embeddings
        frames.append(df)
    if not frames:
        print(f"[no test_predictions.csv found under] {base}")
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


_COORDS_CACHE: dict[str, pd.DataFrame] = {}


def get_patch_coordinates(wsi: str) -> pd.DataFrame:
    if wsi not in _COORDS_CACHE:
        path = P.XENIUM_PROCESSED_OUTPUT_ROOT / wsi / "patch_coordinates.h5"
        with h5py.File(path, "r") as f:
            x_start, y_start = f["x_start"][:], f["y_start"][:]
            raw_ids = f["cell_id"][:]
        cell_id = np.array([c.decode() if isinstance(c, bytes) else c for c in raw_ids])
        _COORDS_CACHE[wsi] = pd.DataFrame({"cell_id": cell_id, "x_centroid_px": x_start + 112, "y_centroid_px": y_start + 112})
    return _COORDS_CACHE[wsi]


def attach_pixel_coords(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = []
    for wsi, group in df.groupby("wsi"):
        coords = get_patch_coordinates(wsi).set_index("cell_id")[["x_centroid_px", "y_centroid_px"]]
        out.append(group.join(coords, on="cell_id"))
    return pd.concat(out, ignore_index=True)


_NEIGHBOR_MAJORITY_CACHE: dict[tuple, tuple[frozenset, pd.Series]] = {}


def neighbor_majority_predictions(df_with_coords: pd.DataFrame, radius_px: float,
                                   label_col: str = "true_label") -> pd.Series:
    preds = pd.Series(index=df_with_coords.index, dtype=object)
    for wsi, group in df_with_coords.groupby("wsi"):
        cache_key = (wsi, radius_px, label_col)
        cell_ids = frozenset(group["cell_id"])
        cached = _NEIGHBOR_MAJORITY_CACHE.get(cache_key)
        if cached is not None and cached[0] == cell_ids:
            by_cell_id = cached[1]
        else:
            xy = group[["x_centroid_px", "y_centroid_px"]].values
            tree = cKDTree(xy)
            neighbor_lists = tree.query_ball_point(xy, r=radius_px)
            labels = group[label_col].values
            result = pd.Series(index=group.index, dtype=object)
            for local_i, (row_idx, neighbors) in enumerate(zip(group.index, neighbor_lists)):
                others = [n for n in neighbors if n != local_i]
                if not others:
                    continue
                vals, counts = np.unique(labels[others], return_counts=True)
                result.loc[row_idx] = vals[np.argmax(counts)]
            by_cell_id = pd.Series(result.values, index=group["cell_id"].values)
            _NEIGHBOR_MAJORITY_CACHE[cache_key] = (cell_ids, by_cell_id)
        preds.loc[group.index] = by_cell_id.reindex(group["cell_id"].values).values
    return preds


def evaluate_neighbor_baseline(df_with_coords: pd.DataFrame, radii_px=tuple(NEIGHBOR_RADII_PX)) -> pd.DataFrame:
    rows = []
    for r in radii_px:
        pred = neighbor_majority_predictions(df_with_coords, r)
        mask = pred.notna()
        rows.append({
            "radius_px": r, "coverage": mask.mean(),
            "macro_f1": f1_score(df_with_coords.loc[mask, "true_label"], pred[mask], average="macro", zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(df_with_coords.loc[mask, "true_label"], pred[mask]),
            "accuracy": (df_with_coords.loc[mask, "true_label"].values == pred[mask].values).mean(),
        })
    rows.append({
        "radius_px": "model (own prediction)", "coverage": 1.0,
        "macro_f1": f1_score(df_with_coords["true_label"], df_with_coords["predicted_label"], average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(df_with_coords["true_label"], df_with_coords["predicted_label"]),
        "accuracy": df_with_coords["correct"].mean(),
    })
    return pd.DataFrame(rows)


for _spec in NEIGHBOR_TEST_PREDICTIONS:
    _df = load_test_predictions(CLASSIFIER_ROOTS["colon"], **_spec)
    if _df.empty:
        continue
    _df_coords = attach_pixel_coords(_df)
    print(f"\n=== {_spec['model_name']} | {_spec['embeddings']}: neighbor-majority baseline vs. model ===")
    show_table(evaluate_neighbor_baseline(_df_coords).round(4),
               f"neighbor_baseline_{_spec['model_name']}_{_spec['embeddings']}")


## 5. Probing: `cell_count` / `cell_density`, central vs. CLS, 100-resized vs. native

4 models \u00d7 2 resolutions (100px-resized, native/224) \u00d7 2 tokens (central, CLS)
\u00d7 2 probes (`cell_count`, `cell_density`) \u00d7 3 context windows (224/448/1344px,
`configs/probing_colon.yaml`'s `probes.cell_count`/`cell_density.context_px`) --
`test_selection_metric` (R\u00b2). Raw table first (long format, mean \u00b1 std across
LOSO splits); diff table is `central - CLS`, **native resolution only**, paired
per split.

In [ ]:
def build_probe_raw_table(models: list[str] = FOUNDATION_MODELS,
                           probes: list[str] = CONTEXT_PROBES_OF_INTEREST,
                           context_px_list: list[int] = CONTEXT_PX_LIST):
    rows = []
    for model in models:
        label = MODEL_LABELS.get(model, model)
        resized_model = RESOLUTION_FAMILIES_BY_NATIVE.get(model)
        for res_label, res_model in (("native", model), ("100resized", resized_model)):
            if res_model is None:
                continue
            for token in (TOKEN_CENTRAL, TOKEN_CLS):
                for probe in probes:
                    for cpx in context_px_list:
                        pkey = probe_key(probe, cpx)
                        m, s, n = mean_std_metric_probe(PROBE_DF, res_model, token, pkey)
                        rows.append({"model": label, "resolution": res_label, "token": TOKEN_LABELS.get(token, token),
                                     "probe": probe, "context_px": cpx, "test_r2_mean": m, "test_r2_std": s, "n_splits": n})
    return pd.DataFrame(rows)


def build_probe_diff_table(models: list[str] = FOUNDATION_MODELS,
                            probes: list[str] = CONTEXT_PROBES_OF_INTEREST,
                            context_px_list: list[int] = CONTEXT_PX_LIST):
    rows = []
    for model in models:
        for probe in probes:
            for cpx in context_px_list:
                pkey = probe_key(probe, cpx)
                d_mean, d_std, d_n = paired_diff_probe(PROBE_DF, model, TOKEN_CENTRAL, model, TOKEN_CLS, pkey)
                rows.append({"model": MODEL_LABELS.get(model, model), "probe": probe, "context_px": cpx,
                             "central_minus_cls_mean": d_mean, "central_minus_cls_std": d_std, "n_splits": d_n})
    return pd.DataFrame(rows)


print(f"[colon] expected {EXPECTED_SPLITS_PROBING} LOSO splits per condition.")
_probe_raw_df = build_probe_raw_table()
_n_missing = (_probe_raw_df["n_splits"] == 0).sum()
if _n_missing:
    print(f"[warning] {_n_missing}/{len(_probe_raw_df)} (model, resolution, token, probe, context_px) rows have 0 fitted splits.")
show_table(_probe_raw_df.round(4), "probe_cellcount_density_raw")

_probe_diff_df = build_probe_diff_table()
show_table(_probe_diff_df.round(4), "probe_cellcount_density_diff_central_minus_cls")


## 6. Probing: Phikon-v2 `area` probe, masked vs. unmasked (central tokens)

`area`: the probed cell's own Xenium boundary-polygon area (log1p-regressed,
independent of any context window). `PhikonV2_448` (unmasked) vs.
`PhikonV2_448_masked` (central 3x3 tokens masked at inference), both read via the
central token.

In [ ]:
def build_mask_probe_tables():
    unmasked, masked = MASK_COMPARISON["unmasked"], MASK_COMPARISON["masked"]
    m_u, s_u, n_u = mean_std_metric_probe(PROBE_DF, unmasked, TOKEN_CENTRAL, CELL_LEVEL_PROBE)
    m_m, s_m, n_m = mean_std_metric_probe(PROBE_DF, masked, TOKEN_CENTRAL, CELL_LEVEL_PROBE)
    raw = pd.DataFrame([
        {"condition": "unmasked", "model_name": unmasked, "test_r2_mean": m_u, "test_r2_std": s_u, "n_splits": n_u},
        {"condition": "masked",   "model_name": masked,   "test_r2_mean": m_m, "test_r2_std": s_m, "n_splits": n_m},
    ])
    d_mean, d_std, d_n = paired_diff_probe(PROBE_DF, unmasked, TOKEN_CENTRAL, masked, TOKEN_CENTRAL, CELL_LEVEL_PROBE)
    diff = pd.DataFrame([{"probe": CELL_LEVEL_PROBE, "comparison": "unmasked - masked",
                          "diff_mean": d_mean, "diff_std": d_std, "n_splits": d_n}])
    return raw, diff


print(f"[colon] expected {EXPECTED_SPLITS_PROBING} LOSO splits per condition.")
_mask_probe_raw, _mask_probe_diff = build_mask_probe_tables()
show_table(_mask_probe_raw.round(4), "phikon_area_probe_masked_vs_unmasked_raw")
show_table(_mask_probe_diff.round(4), "phikon_area_probe_masked_vs_unmasked_diff")


## 7. Classification: Phikon-v2 colon classification, masked vs. unmasked (central tokens)

Same masked/unmasked Phikon-v2 pair as section 6, but for the colon cell-type
**classification** task (`outputs_classifier_colon`), central token, macro-F1.

In [ ]:
def build_mask_classifier_tables(dataset: str = "colon"):
    unmasked, masked = MASK_COMPARISON["unmasked"], MASK_COMPARISON["masked"]
    m_u, s_u, n_u = mean_std_n(CLASSIFIER_DF, dataset, unmasked, TOKEN_CENTRAL)
    m_m, s_m, n_m = mean_std_n(CLASSIFIER_DF, dataset, masked, TOKEN_CENTRAL)
    raw = pd.DataFrame([
        {"condition": "unmasked", "model_name": unmasked, "macro_f1_mean": m_u, "macro_f1_std": s_u, "n_splits": n_u},
        {"condition": "masked",   "model_name": masked,   "macro_f1_mean": m_m, "macro_f1_std": s_m, "n_splits": n_m},
    ])
    d_mean, d_std, d_n = paired_diff(CLASSIFIER_DF, dataset, unmasked, TOKEN_CENTRAL, masked, TOKEN_CENTRAL)
    diff = pd.DataFrame([{"comparison": "unmasked - masked", "diff_mean": d_mean, "diff_std": d_std, "n_splits": d_n}])
    return raw, diff


print(f"[colon] expected {EXPECTED_SPLITS['colon']} LOSO splits per condition.")
_mask_clf_raw, _mask_clf_diff = build_mask_classifier_tables()
show_table(_mask_clf_raw.round(4), "phikon_colon_classification_masked_vs_unmasked_raw")
show_table(_mask_clf_diff.round(4), "phikon_colon_classification_masked_vs_unmasked_diff")


## 8. Classification: UNI2 cross-cancer, central vs. default tokens

`UNI2_specific_tokens_folder` on the cross-cancer classification task:
`embeddings=central` (the 4-corner average) vs. `embeddings=default` (average of
every `embeddings_*` key found in the h5 file -- mixes in CLS/boundary tokens too).
**Assumption flagged**: this uses the same `UNI2_specific_tokens_folder` name as
the colon runs -- if cross-cancer's UNI2 native extraction actually lives under a
different folder (e.g. `UNI2_896`), the `n_splits` columns below will show 0 and
you'll need to point `_UNI2_CROSS_CANCER_MODEL` at the right name.

In [ ]:
_UNI2_CROSS_CANCER_MODEL = "UNI2_specific_tokens_folder"  # ADJUST if different from the colon folder


def build_uni2_default_vs_central_table(dataset: str = "cross_cancer", model: str = _UNI2_CROSS_CANCER_MODEL):
    m_c, s_c, n_c = mean_std_n(CLASSIFIER_DF, dataset, model, TOKEN_CENTRAL)
    m_d, s_d, n_d = mean_std_n(CLASSIFIER_DF, dataset, model, TOKEN_DEFAULT)
    raw = pd.DataFrame([
        {"token": "central", "macro_f1_mean": m_c, "macro_f1_std": s_c, "n_splits": n_c},
        {"token": "default", "macro_f1_mean": m_d, "macro_f1_std": s_d, "n_splits": n_d},
    ])
    d_mean, d_std, d_n = paired_diff(CLASSIFIER_DF, dataset, model, TOKEN_DEFAULT, model, TOKEN_CENTRAL)
    diff = pd.DataFrame([{"comparison": "default - central", "diff_mean": d_mean, "diff_std": d_std, "n_splits": d_n}])
    return raw, diff


print(f"[cross_cancer] expected {EXPECTED_SPLITS['cross_cancer']} LOSO splits per condition.")
_uni2_dc_raw, _uni2_dc_diff = build_uni2_default_vs_central_table()
show_table(_uni2_dc_raw.round(4), "uni2_cross_cancer_default_vs_central_raw")
show_table(_uni2_dc_diff.round(4), "uni2_cross_cancer_default_vs_central_diff")


## 9. Masking-drift vs. biological difference: top/bottom-drift cell distributions

From `analyse_probing_results.ipynb`'s "Top-drift vs. bottom-drift cells" section
(group-contrast version of its drift-vs-biological-difference correlation test).
**Enhanced here**: instead of a boxplot with back-to-back marginal histograms, both
groups' distributions are drawn overlaid on the same axes, together with the
distribution over *all* cells (not just the two extreme groups), plus the same
WSI-cluster-bootstrap statistical test annotated directly on the plot.

Re-reads raw `embeddings_dataset.h5` / `patch_coordinates.h5` / `adata.h5ad` files
directly (bypasses the probing pipeline's own subsampling) -- needs a masked/unmasked
model pair to already be probed (`PROBE_DF` above) plus `DATASETS_ROOT` /
`XENIUM_PROCESSED_OUTPUT_ROOT` set.

In [ ]:
import anndata as ad  # noqa: E402
from src.python.probing.data.cell_population import PopulationCache  # noqa: E402
from src.python.probing.probes.base import group_ranges  # noqa: E402

_known_runs = set(zip(PROBE_DF["model_name"], PROBE_DF["correction_name"])) if not PROBE_DF.empty else set()
MASKED_UNMASKED_PAIRS = sorted(
    (masked[: -len("_masked")], masked, correction)
    for masked, correction in _known_runs
    if masked.endswith("_masked") and (masked[: -len("_masked")], correction) in _known_runs
)
print("Detected masked/unmasked pairs:", MASKED_UNMASKED_PAIRS)

TOP_BOTTOM_DRIFT_FRAC = 0.05   # top/bottom 5% of cells by drift (pooled across WSIs)
DRIFT_CONTEXT_PX = 448          # matches these runs' size_side=448 field of view
DRIFT_MIN_NEIGHBORS = 3


In [ ]:
def _decode_str_array(arr: np.ndarray) -> np.ndarray:
    if len(arr) and isinstance(arr[0], (bytes, bytearray)):
        return np.array([x.decode("utf-8") for x in arr])
    return arr


def _load_raw_cell_embeddings(base_path, wsi_name: str, embeddings_datasets: list[str]):
    with h5py.File(os.path.join(base_path, wsi_name, "embeddings_dataset.h5"), "r") as f:
        datasets = embeddings_datasets or [k.split("embeddings_")[1] for k in f.keys() if k.startswith("embeddings_")]
        embeddings = np.mean([f[f"embeddings_{k}"][()] for k in datasets], axis=0).astype(np.float64)
        cell_ids = _decode_str_array(f["cell_ids"][()])
    return cell_ids, embeddings


def _cosine_distance(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_n = a / np.clip(np.linalg.norm(a, axis=1, keepdims=True), 1e-12, None)
    b_n = b / np.clip(np.linalg.norm(b, axis=1, keepdims=True), 1e-12, None)
    return 1.0 - np.sum(a_n * b_n, axis=1)


def compute_drift(unmasked_model: str, masked_model: str, correction_name: str,
                   embeddings_datasets: list[str]) -> pd.DataFrame:
    unmasked_base = os.path.join(P.DATASETS_ROOT, f"{unmasked_model}_h5", correction_name)
    masked_base = os.path.join(P.DATASETS_ROOT, f"{masked_model}_h5", correction_name)
    wsi_names = sorted(set(os.listdir(unmasked_base)) & set(os.listdir(masked_base)))
    if not wsi_names:
        raise ValueError(f"No shared WSI directories between {unmasked_base!r} and {masked_base!r}")
    rows = []
    for wsi_name in wsi_names:
        ids_u, emb_u = _load_raw_cell_embeddings(unmasked_base, wsi_name, embeddings_datasets)
        ids_m, emb_m = _load_raw_cell_embeddings(masked_base, wsi_name, embeddings_datasets)
        common, iu, im = np.intersect1d(ids_u, ids_m, return_indices=True)
        if len(common) == 0:
            print(f"[warning] {wsi_name}: no cell_ids shared between {unmasked_model} and {masked_model}")
            continue
        drift = _cosine_distance(emb_u[iu], emb_m[im])
        rows.append(pd.DataFrame({"wsi_name": wsi_name, "cell_id": common, "drift": drift}))
    return pd.concat(rows, ignore_index=True)


def _pearson_corr(a, b):
    a, b = a - a.mean(), b - b.mean()
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan


def _cosine_dist_vec(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(1.0 - np.dot(a, b) / denom) if denom > 0 else np.nan


def _xenium_barcode(cell_id: str) -> str:
    return cell_id.split("_")[-1]


def compute_neighbor_type_stats(df: pd.DataFrame, mapping_name: str, context_px: float) -> pd.DataFrame:
    mapping = mapping_factory(mapping_name)
    pop_cache = PopulationCache(P.XENIUM_PROCESSED_OUTPUT_ROOT, mapping)
    half = context_px / 2.0
    n = len(df)
    position_found = np.zeros(n, dtype=bool)
    n_neighbors = np.zeros(n, dtype=np.int64)
    heterotypic_frac = np.full(n, np.nan)

    wsi_names = df["wsi_name"].to_numpy()
    cell_ids = df["cell_id"].to_numpy()
    order = np.argsort(wsi_names, kind="stable")
    sorted_wsi = wsi_names[order]
    for start, end in group_ranges(sorted_wsi):
        wsi_name = sorted_wsi[start]
        pop = pop_cache.get(wsi_name)
        local_idx = order[start:end]
        rows = np.array([pop.id_to_row.get(cid, -1) for cid in cell_ids[local_idx]])
        valid = rows >= 0
        if not valid.any():
            continue
        v_idx, v_rows = local_idx[valid], rows[valid]
        position_found[v_idx] = True
        neighbor_lists = pop.query_windows(pop.x[v_rows], pop.y[v_rows], half)
        for li, row, neighbors in zip(v_idx.tolist(), v_rows.tolist(), neighbor_lists):
            neighbors = np.asarray(neighbors, dtype=np.int64)
            neighbors = neighbors[neighbors != row]
            n_neighbors[li] = len(neighbors)
            if len(neighbors) > 0:
                same = np.count_nonzero(pop.class_idx[neighbors] == pop.class_idx[row])
                heterotypic_frac[li] = 1.0 - same / len(neighbors)

    out = df.copy()
    out["position_found"] = position_found
    out["n_type_neighbors"] = n_neighbors
    out["heterotypic_frac"] = heterotypic_frac
    out["isolated"] = position_found & (n_neighbors == 0)
    return out


def _load_adata_population(wsi_name: str, target_sum: float = 100.0) -> dict:
    path = os.path.join(str(P.XENIUM_PROCESSED_OUTPUT_ROOT), wsi_name, "adata.h5ad")
    adata = ad.read_h5ad(path)
    counts = adata.layers["counts"]
    counts = np.asarray(counts.todense() if hasattr(counts, "todense") else counts, dtype=np.float64)
    lib_size = counts.sum(axis=1, keepdims=True)
    lib_size[lib_size == 0] = 1.0
    expr = np.log1p(counts / lib_size * target_sum)
    cell_id = _decode_str_array(np.asarray(adata.obs["cell_id"].to_numpy()))
    x = adata.obs["x_centroid"].to_numpy(dtype=np.float64)
    y = adata.obs["y_centroid"].to_numpy(dtype=np.float64)
    tree = cKDTree(np.column_stack([x, y]))
    return {"x": x, "y": y, "expr": expr, "tree": tree, "id_to_row": {cid: i for i, cid in enumerate(cell_id)}}


_ADATA_POP_CACHE: dict[str, dict] = {}


def _get_adata_population(wsi_name: str) -> dict:
    if wsi_name not in _ADATA_POP_CACHE:
        _ADATA_POP_CACHE[wsi_name] = _load_adata_population(wsi_name)
    return _ADATA_POP_CACHE[wsi_name]


def compute_expression_divergence(df: pd.DataFrame, context_px: float, min_neighbors: int) -> pd.DataFrame:
    n = len(df)
    cos_dist = np.full(n, np.nan)
    half = context_px / 2.0
    wsi_names = df["wsi_name"].to_numpy()
    cell_ids = df["cell_id"].to_numpy()
    order = np.argsort(wsi_names, kind="stable")
    sorted_wsi = wsi_names[order]
    for start, end in group_ranges(sorted_wsi):
        wsi_name = sorted_wsi[start]
        pop = _get_adata_population(wsi_name)
        local_idx = order[start:end]
        rows = np.array([pop["id_to_row"].get(_xenium_barcode(cid), -1) for cid in cell_ids[local_idx]])
        valid = rows >= 0
        if not valid.any():
            continue
        v_idx, v_rows = local_idx[valid], rows[valid]
        neighbor_lists = pop["tree"].query_ball_point(
            np.column_stack([pop["x"][v_rows], pop["y"][v_rows]]), r=half, p=np.inf, workers=-1
        )
        for li, row, neighbors in zip(v_idx.tolist(), v_rows.tolist(), neighbor_lists):
            neighbors = np.asarray(neighbors, dtype=np.int64)
            neighbors = neighbors[neighbors != row]
            if len(neighbors) < min_neighbors:
                continue
            neighborhood_mean = pop["expr"][neighbors].mean(axis=0)
            cos_dist[li] = _cosine_dist_vec(pop["expr"][row], neighborhood_mean)
    out = df.copy()
    out["expr_cosine_dist_to_neighbors"] = cos_dist
    return out


def cluster_bootstrap_group_diff(values: np.ndarray, mask_a: np.ndarray, mask_b: np.ndarray, groups: np.ndarray,
                                  n_boot: int = 2000, seed: int = 0) -> dict:
    values, mask_a, mask_b, groups = np.asarray(values), np.asarray(mask_a), np.asarray(mask_b), np.asarray(groups)
    valid = np.isfinite(values) & (mask_a | mask_b)
    values, mask_a, mask_b, groups = values[valid], mask_a[valid], mask_b[valid], groups[valid]
    wsi_u = np.unique(groups)
    rng = np.random.default_rng(seed)
    if mask_a.sum() == 0 or mask_b.sum() == 0:
        return {"n_a": int(mask_a.sum()), "n_b": int(mask_b.sum()), "median_diff": np.nan,
                "ci_lo": np.nan, "ci_hi": np.nan, "mannwhitney_p_naive_iid": np.nan}
    obs = float(np.median(values[mask_a]) - np.median(values[mask_b]))
    boot = []
    for _ in range(n_boot):
        sampled = rng.choice(wsi_u, size=len(wsi_u), replace=True)
        idx = np.concatenate([np.flatnonzero(groups == w) for w in sampled])
        d, a, b = values[idx], mask_a[idx], mask_b[idx]
        if a.any() and b.any():
            boot.append(np.median(d[a]) - np.median(d[b]))
    boot = np.array(boot)
    if len(boot) == 0:
        ci_lo = ci_hi = np.nan
    else:
        ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
    _, u_p = stats.mannwhitneyu(values[mask_a], values[mask_b], alternative="greater")
    return {"n_a": int(mask_a.sum()), "n_b": int(mask_b.sum()), "median_diff": obs,
            "ci_lo": float(ci_lo), "ci_hi": float(ci_hi), "mannwhitney_p_naive_iid": float(u_p)}


def top_bottom_drift_masks(df: pd.DataFrame, frac: float = TOP_BOTTOM_DRIFT_FRAC):
    drift = df["drift"].to_numpy()
    valid = np.isfinite(drift)
    lo_thresh = np.nanpercentile(drift[valid], frac * 100)
    hi_thresh = np.nanpercentile(drift[valid], (1 - frac) * 100)
    bottom_mask = valid & (drift <= lo_thresh)
    top_mask = valid & (drift >= hi_thresh)
    return top_mask, bottom_mask


def plot_group_distributions_with_all(df: pd.DataFrame, value_col: str, xlabel: str,
                                       mask_top: np.ndarray, mask_bottom: np.ndarray,
                                       label_top: str, label_bottom: str, title: str,
                                       bins: int = 40, clip_pct=(0.5, 99.5)):
    x_all = df[value_col].to_numpy(dtype=float)
    x_all = x_all[np.isfinite(x_all)]
    top = df.loc[mask_top, value_col].dropna().to_numpy()
    bottom = df.loc[mask_bottom, value_col].dropna().to_numpy()
    lo, hi = (np.percentile(x_all, clip_pct) if clip_pct else (x_all.min(), x_all.max()))
    edges = np.linspace(lo, hi, bins + 1)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(x_all, bins=edges, density=True, color="0.75", alpha=0.6, label=f"all cells (n={len(x_all)})")
    ax.hist(bottom, bins=edges, density=True, histtype="step", linewidth=1.8, color="C0",
            label=f"{label_bottom} (n={len(bottom)})")
    ax.hist(top, bins=edges, density=True, histtype="step", linewidth=1.8, color="C1",
            label=f"{label_top} (n={len(top)})")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("density")
    ax.legend(fontsize=8, frameon=False)
    ax.grid(axis="y", alpha=0.3)

    stat = cluster_bootstrap_group_diff(df[value_col].to_numpy(), np.asarray(mask_top), np.asarray(mask_bottom),
                                         df["wsi_name"].to_numpy())
    annotation = (f"median diff (top - bottom) = {stat['median_diff']:.3g}\n"
                  f"95% WSI-cluster bootstrap CI: [{stat['ci_lo']:.3g}, {stat['ci_hi']:.3g}]\n"
                  f"Mann-Whitney p (naive, iid) = {stat['mannwhitney_p_naive_iid']:.3g}")
    ax.text(0.98, 0.98, annotation, transform=ax.transAxes, ha="right", va="top", fontsize=8,
            bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
    ax.set_title(title)
    fig.tight_layout()
    save_fig(fig, f"drift_group_distributions_{value_col}")
    plt.show()
    return stat


In [ ]:
if MASKED_UNMASKED_PAIRS:
    DRIFT_PAIR = MASKED_UNMASKED_PAIRS[0]
    print(f"Using drift pair: {DRIFT_PAIR}")
    drift_df = compute_drift(*DRIFT_PAIR, TOKEN_CENTRAL_KEYS)
    print(f"{len(drift_df)} matched cells across {drift_df['wsi_name'].nunique()} WSIs")

    _mapping_rows = PROBE_DF.loc[PROBE_DF["model_name"] == DRIFT_PAIR[0], "mapping"]
    mapping_name = _mapping_rows.iloc[0] if not _mapping_rows.empty else "simplified_broad"
    analysis_df = compute_neighbor_type_stats(drift_df, mapping_name, DRIFT_CONTEXT_PX)
    analysis_df = compute_expression_divergence(analysis_df, DRIFT_CONTEXT_PX, DRIFT_MIN_NEIGHBORS)
    print(f"{len(analysis_df)} matched cells total; {int(analysis_df['isolated'].sum())} isolated "
          f"(0 neighbours within {DRIFT_CONTEXT_PX}px).")

    top_mask, bottom_mask = top_bottom_drift_masks(analysis_df)
    print(f"top-drift: n={int(top_mask.sum())}, bottom-drift: n={int(bottom_mask.sum())} "
          f"(top/bottom {TOP_BOTTOM_DRIFT_FRAC:.0%} of {int(np.isfinite(analysis_df['drift']).sum())} cells with a drift value)")

    for value_col, xlabel in [("heterotypic_frac", "fraction of neighbours with a different mapped cell_type"),
                              ("expr_cosine_dist_to_neighbors", "1 - cosine similarity to neighbourhood mean Xenium expression")]:
        plot_group_distributions_with_all(
            analysis_df, value_col, xlabel, top_mask, bottom_mask,
            f"top {TOP_BOTTOM_DRIFT_FRAC:.0%} drift", f"bottom {TOP_BOTTOM_DRIFT_FRAC:.0%} drift",
            f"{value_col} by drift group, vs. all cells",
        )
else:
    print("[skip] no masked/unmasked model pair found in PROBE_DF -- see MASKED_UNMASKED_PAIRS above.")


## 10. RNA-context divergence vs. masking disagreement: distributions

From `saved_cells_deep_dive.ipynb`'s \u00a78 ("RNA expression vs. context, for the
masking disagreement set"): for cells correct under both Phikon-v2 masked and
unmasked ("correct in both") vs. cells correct unmasked but wrong when masked
("correct unmasked, wrong masked"), cosine distance between each cell's
log1p-normalized Xenium expression and the mean expression of its `k=30` nearest
spatial neighbours. **Enhanced here**: overlaid distributions for the two groups
(instead of a plain boxplot) plus both the original Mann-Whitney test *and* a
WSI-cluster bootstrap (same test as section 9) annotated on the plot -- cells
within a WSI aren't independent draws, so the cluster bootstrap is the more
defensible of the two; Mann-Whitney is kept only as the original reference point.

Uses `PhikonV2_448` / `PhikonV2_448_masked` `test_predictions.csv`, read via
`TOKEN_CENTRAL` for consistency with the rest of this notebook -- swap to
`"nucleus"` below if that's what was actually saved for these two colon runs
(the original notebook's `SAVED_PREDICTIONS` used `nucleus`, colon's classifier
default embeddings segment at the time it was written).

In [ ]:
import scanpy as sc  # noqa: E402

RNA_DIVERGENCE_EMBEDDINGS = TOKEN_CENTRAL  # or "nucleus" -- see markdown above
RNA_DIVERGENCE_K_NEIGHBORS = 30


def align_conditions(df_a: pd.DataFrame, df_b: pd.DataFrame, label_a: str, label_b: str) -> pd.DataFrame:
    keep = ["cell_id", "wsi", "true_label", "predicted_label", "correct"]
    a = df_a[keep].rename(columns={c: f"{c}__{label_a}" for c in keep if c not in ("cell_id", "wsi")})
    b = df_b[keep].rename(columns={c: f"{c}__{label_b}" for c in keep if c not in ("cell_id", "wsi")})
    merged = a.merge(b, on=["cell_id", "wsi"], how="inner")
    assert (merged[f"true_label__{label_a}"] == merged[f"true_label__{label_b}"]).all(), \
        "true_label disagrees between conditions for a shared cell_id -- check both used the same mapping"
    merged["true_label"] = merged[f"true_label__{label_a}"]
    correct_a, correct_b = merged[f"correct__{label_a}"], merged[f"correct__{label_b}"]
    merged["quadrant"] = np.select(
        [correct_a & correct_b, (~correct_a) & (~correct_b), correct_a & (~correct_b), (~correct_a) & correct_b],
        ["both_correct", "both_wrong", f"{label_a}_only_correct", f"{label_b}_only_correct"],
    )
    return merged


_ADATA_LOGNORM_CACHE: dict[str, "ad.AnnData"] = {}


def load_adata_lognorm(wsi: str) -> "ad.AnnData":
    if wsi not in _ADATA_LOGNORM_CACHE:
        path = P.XENIUM_PROCESSED_OUTPUT_ROOT / wsi / "adata.h5ad"
        adata = ad.read_h5ad(path)
        adata.X = adata.layers["counts"].copy()
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        _ADATA_LOGNORM_CACHE[wsi] = adata
    return _ADATA_LOGNORM_CACHE[wsi]


def rna_context_divergence(wsi: str, k_neighbors: int = RNA_DIVERGENCE_K_NEIGHBORS) -> pd.DataFrame:
    adata = load_adata_lognorm(wsi)
    coords = get_patch_coordinates(wsi).set_index("cell_id")
    common = coords.index.intersection(adata.obs.index)
    if len(common) < k_neighbors + 1:
        return pd.DataFrame(columns=["divergence"])
    barcode_to_row = {b: i for i, b in enumerate(adata.obs.index.values)}
    rows = [barcode_to_row[c] for c in common]
    X = adata.X[rows]
    X = np.asarray(X.todense()) if hasattr(X, "todense") else np.asarray(X)
    xy = coords.loc[common, ["x_centroid_px", "y_centroid_px"]].values
    tree = cKDTree(xy)
    _, idx = tree.query(xy, k=k_neighbors + 1)
    neighbor_mean = X[idx[:, 1:]].mean(axis=1)
    own_norm = np.linalg.norm(X, axis=1)
    neigh_norm = np.linalg.norm(neighbor_mean, axis=1)
    denom = own_norm * neigh_norm
    cos_sim = np.divide((X * neighbor_mean).sum(axis=1), denom,
                         out=np.full(len(common), np.nan), where=denom > 0)
    return pd.DataFrame({"divergence": 1 - cos_sim}, index=pd.Index(common, name="cell_id"))


def attach_rna_divergence(df: pd.DataFrame, k_neighbors: int = RNA_DIVERGENCE_K_NEIGHBORS) -> pd.DataFrame:
    if df.empty:
        return df
    out = []
    for wsi, group in df.groupby("wsi"):
        div = rna_context_divergence(wsi, k_neighbors)
        out.append(group.join(div, on="cell_id"))
    return pd.concat(out, ignore_index=True)


def plot_two_group_distributions(group_a: np.ndarray, group_b: np.ndarray, label_a: str, label_b: str,
                                  xlabel: str, title: str, groups_for_bootstrap: np.ndarray = None,
                                  bins: int = 40) -> dict:
    pooled = np.concatenate([group_a, group_b])
    lo, hi = np.percentile(pooled, [0.5, 99.5])
    edges = np.linspace(lo, hi, bins + 1)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(group_b, bins=edges, density=True, histtype="step", linewidth=1.8, color="C0", label=f"{label_b} (n={len(group_b)})")
    ax.hist(group_a, bins=edges, density=True, histtype="step", linewidth=1.8, color="C1", label=f"{label_a} (n={len(group_a)})")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("density")
    ax.legend(fontsize=8, frameon=False)
    ax.grid(axis="y", alpha=0.3)

    _, mw_p = mannwhitneyu(group_a, group_b, alternative="two-sided")
    annotation = f"Mann-Whitney p (naive, iid) = {mw_p:.3g}"
    if groups_for_bootstrap is not None:
        values = pooled
        mask_a = np.concatenate([np.ones(len(group_a), dtype=bool), np.zeros(len(group_b), dtype=bool)])
        mask_b = ~mask_a
        stat = cluster_bootstrap_group_diff(values, mask_a, mask_b, groups_for_bootstrap)
        annotation = (f"median diff ({label_a} - {label_b}) = {stat['median_diff']:.3g}\n"
                      f"95% WSI-cluster bootstrap CI: [{stat['ci_lo']:.3g}, {stat['ci_hi']:.3g}]\n" + annotation)
    ax.text(0.98, 0.98, annotation, transform=ax.transAxes, ha="right", va="top", fontsize=8,
            bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
    ax.set_title(title)
    fig.tight_layout()
    save_fig(fig, "rna_divergence_broken_by_masking_distributions")
    plt.show()
    return {"mannwhitney_p": float(mw_p)}


In [ ]:
_masked_preds = load_test_predictions(CLASSIFIER_ROOTS["colon"], MASK_COMPARISON["masked"], RNA_DIVERGENCE_EMBEDDINGS)
_unmasked_preds = load_test_predictions(CLASSIFIER_ROOTS["colon"], MASK_COMPARISON["unmasked"], RNA_DIVERGENCE_EMBEDDINGS)

if not _masked_preds.empty and not _unmasked_preds.empty:
    unmasked_div = attach_rna_divergence(_unmasked_preds)
    aligned_mask_div = align_conditions(unmasked_div, _masked_preds, "unmasked", "masked").merge(
        unmasked_div[["cell_id", "wsi", "divergence"]], on=["cell_id", "wsi"], how="left"
    )
    both_correct = aligned_mask_div.loc[aligned_mask_div["quadrant"] == "both_correct", ["divergence", "wsi"]].dropna()
    broken_by_masking = aligned_mask_div.loc[aligned_mask_div["quadrant"] == "unmasked_only_correct", ["divergence", "wsi"]].dropna()

    if len(both_correct) > 5 and len(broken_by_masking) > 5:
        pooled_wsi = pd.concat([both_correct["wsi"], broken_by_masking["wsi"]]).to_numpy()
        plot_two_group_distributions(
            broken_by_masking["divergence"].to_numpy(), both_correct["divergence"].to_numpy(),
            "correct unmasked, wrong masked", "correct in both",
            "RNA-context divergence (1 - cosine sim to k=30 neighbor mean)",
            "Do cells that break under masking show higher RNA-context divergence?",
            groups_for_bootstrap=pooled_wsi,
        )
    else:
        print(f"[skip] too few cells in one group (both_correct n={len(both_correct)}, "
              f"broken_by_masking n={len(broken_by_masking)})")
else:
    print("[skip] masked/unmasked PhikonV2 test_predictions not both loaded -- see [missing] above.")


## 11. Multicell UNI2 (colon): performance vs. forward passes / reduction factor

`UNI2_448_112_multicell` / `UNI2_448_224_multicell` / `UNI2_448_448_multicell`
(one per central-window size -- see `README.md`'s `central_size_x`/`central_size_y`),
read via `cell_token` (multicell's single per-cell token, no CLS/corner split),
macro-F1 on the colon classification task. Dashed line = the CLS baseline
(`UNI2_specific_tokens_folder | cls`, plain per-cell UNI2).

**`FORWARD_PASSES` in CONFIG has `None` placeholders** -- fill in the real
per-model forward-pass counts there (including the plain per-cell UNI2 baseline,
needed as the reduction-factor denominator: `reduction_factor =
FORWARD_PASSES[CLS_BASELINE_MODEL] / FORWARD_PASSES[model]`) before running this
section; rows with a missing count are skipped from the x=forward-passes /
x=reduction-factor plots (but still shown in the table) and reported below.

In [ ]:
def build_multicell_table(dataset: str = "colon", token: str = MULTICELL_TOKEN):
    baseline_fp = FORWARD_PASSES.get(CLS_BASELINE_MODEL)
    baseline_f1_mean, baseline_f1_std, baseline_f1_n = mean_std_n(CLASSIFIER_DF, dataset, CLS_BASELINE_MODEL, TOKEN_CLS)
    rows = []
    for model in MULTICELL_MODELS:
        fp = FORWARD_PASSES.get(model)
        m, s, n = mean_std_n(CLASSIFIER_DF, dataset, model, token)
        gain_m, gain_s, gain_n = paired_diff(CLASSIFIER_DF, dataset, model, token, CLS_BASELINE_MODEL, TOKEN_CLS)
        reduction = (baseline_fp / fp) if (baseline_fp and fp) else np.nan
        rows.append({"model": MODEL_LABELS.get(model, model), "model_name": model,
                     "num_forward_passes": fp, "reduction_factor": reduction,
                     "macro_f1_mean": m, "macro_f1_std": s, "n_splits": n,
                     "gain_over_cls_baseline_mean": gain_m, "gain_over_cls_baseline_std": gain_s,
                     "gain_n_splits": gain_n})
    df = pd.DataFrame(rows)
    _missing_fp = [m for m in MULTICELL_MODELS + [CLS_BASELINE_MODEL] if FORWARD_PASSES.get(m) is None]
    if _missing_fp:
        print(f"[warning] FORWARD_PASSES missing for: {_missing_fp} -- fill these in in CONFIG. "
              f"num_forward_passes/reduction_factor plots will skip the affected rows.")
    print(f"[colon] expected {EXPECTED_SPLITS['colon']} LOSO splits per condition; "
          f"CLS baseline macro-F1 = {fmt_mean_std(baseline_f1_mean, baseline_f1_std, baseline_f1_n, EXPECTED_SPLITS['colon'])}")
    return df, baseline_f1_mean


_multicell_df, _cls_baseline_f1 = build_multicell_table()
show_table(_multicell_df.round(4), "multicell_uni2_colon")


def plot_multicell(df: pd.DataFrame, x_col: str, y_mean_col: str, y_std_col: str,
                    xlabel: str, ylabel: str, title: str, baseline: float = None, baseline_label: str = "") -> None:
    plot_df = df.dropna(subset=[x_col, y_mean_col])
    if plot_df.empty:
        print(f"[skip plot] no rows with both {x_col!r} and {y_mean_col!r} available.")
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    order = plot_df[x_col].argsort()
    ax.errorbar(plot_df[x_col].to_numpy()[order], plot_df[y_mean_col].to_numpy()[order],
                yerr=plot_df[y_std_col].to_numpy()[order], marker="o", capsize=3, color="#4C72B0")
    for _, row in plot_df.iterrows():
        ax.annotate(row["model"], (row[x_col], row[y_mean_col]), fontsize=7, textcoords="offset points", xytext=(4, 4))
    if baseline is not None and np.isfinite(baseline):
        ax.axhline(baseline, color="black", linestyle="--", linewidth=1.2, label=baseline_label)
        ax.legend(fontsize=8)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    save_fig(fig, title.lower().replace(" ", "_").replace("/", "_"))
    plt.show()


plot_multicell(_multicell_df, "num_forward_passes", "macro_f1_mean", "macro_f1_std",
               "number of forward passes", "macro-F1 (mean \u00b1 std)",
               "Multicell UNI2 (colon): macro-F1 vs. forward passes",
               baseline=_cls_baseline_f1, baseline_label="CLS baseline (UNI2 colon)")

plot_multicell(_multicell_df, "reduction_factor", "macro_f1_mean", "macro_f1_std",
               "reduction factor (forward passes: per-cell / multicell)", "macro-F1 (mean \u00b1 std)",
               "Multicell UNI2 (colon): macro-F1 vs. reduction factor",
               baseline=_cls_baseline_f1, baseline_label="CLS baseline (UNI2 colon)")

plot_multicell(_multicell_df, "num_forward_passes", "gain_over_cls_baseline_mean", "gain_over_cls_baseline_std",
               "number of forward passes", "gain in macro-F1 over CLS baseline (mean \u00b1 std)",
               "Multicell UNI2 (colon): gain over CLS baseline vs. forward passes")

plot_multicell(_multicell_df, "reduction_factor", "gain_over_cls_baseline_mean", "gain_over_cls_baseline_std",
               "reduction factor (forward passes: per-cell / multicell)", "gain in macro-F1 over CLS baseline (mean \u00b1 std)",
               "Multicell UNI2 (colon): gain over CLS baseline vs. reduction factor")


## 12. DeepSpot colon: UNI2 vs. UNI2 (resized), fold-change vs. 30 neighbours

`outputs_deepspot_colon`: `UNI2_specific_tokens_folder` ("UNI2") vs.
`UNI2_56_resized` ("UNI2 resized"), across `num_neighbours` \u2208 {30, 15, 0} --
fold-change (and % decrease) in performance relative to each model's own
30-neighbour point. Metric auto-picked from whatever `*pearson*` column
`load_deepspot_results` found in `results_summary.yaml` (printed above when
`DEEPSPOT_DF` was loaded) -- set `DEEPSPOT_METRIC` below if the auto-pick is wrong.

In [ ]:
DEEPSPOT_METRIC = None  # None = auto-pick a "*mean*pearson*" column, else the first pearson-like column found


def build_deepspot_foldchange_table(df: pd.DataFrame = DEEPSPOT_DF, metric: str = DEEPSPOT_METRIC):
    if df.empty:
        print("[empty] DEEPSPOT_DF is empty -- nothing to compute.")
        return pd.DataFrame()
    pearson_cols = [c for c in df.columns if "pearson" in c.lower()]
    if not pearson_cols:
        print("[warning] no pearson-like column found in results_summary.yaml -- check the actual key names there.")
        return pd.DataFrame()
    metric = metric or next((c for c in pearson_cols if "mean" in c.lower()), pearson_cols[0])
    print(f"[using metric] {metric!r} (available: {pearson_cols})")

    rows = []
    for label, model in DEEPSPOT_MODELS.items():
        sub_model = df[df["model_name"] == model]
        base_vals = sub_model.loc[sub_model["num_neighbours"] == 30, metric].dropna()
        base_mean = base_vals.mean() if len(base_vals) else np.nan
        for n_neighbours in DEEPSPOT_NEIGHBOURS:
            vals = sub_model.loc[sub_model["num_neighbours"] == n_neighbours, metric].dropna()
            mean_v = vals.mean() if len(vals) else np.nan
            std_v = vals.std() if len(vals) > 1 else 0.0
            fold_change = (mean_v / base_mean) if (base_mean and np.isfinite(base_mean) and np.isfinite(mean_v)) else np.nan
            pct_decrease = (1 - fold_change) * 100 if np.isfinite(fold_change) else np.nan
            rows.append({"model": label, "model_name": model, "num_neighbours": n_neighbours,
                         f"{metric}_mean": mean_v, f"{metric}_std": std_v, "n_splits": len(vals),
                         "fold_change_vs_30_neighbours": fold_change, "pct_decrease_vs_30_neighbours": pct_decrease})
    print(f"[colon] expected {EXPECTED_SPLITS_DEEPSPOT_COLON} LOSO splits per condition.")
    return pd.DataFrame(rows)


_deepspot_fc_df = build_deepspot_foldchange_table()
if not _deepspot_fc_df.empty:
    show_table(_deepspot_fc_df.round(4), "deepspot_colon_uni2_foldchange")


## Assumptions to double-check before trusting a number

This notebook was written against documented conventions, not run against real
data (see the top of this notebook). Things worth re-checking once real data is
in place, in order of how likely they are to be wrong:

1. **`_UNI2_CROSS_CANCER_MODEL` (section 8)** and, more generally, whether
   `UNI2_specific_tokens_folder` is really the same folder name for both the
   colon *and* cross-cancer native UNI2 extraction -- flagged explicitly in
   section 8, but every other section using `FOUNDATION_MODELS`/`RESOLUTION_FAMILIES_BY_NATIVE`
   on the `cross_cancer` dataset (section 1, 2) makes the same assumption.
2. **`UNI2_RESIZE_SWEEP`'s 100px "smallest" baseline (section 3)** -- a stand-in
   per your note, not the real smallest-FOV run.
3. **`FORWARD_PASSES` (section 11)** -- all `None` placeholders; fill these in
   before trusting the forward-pass/reduction-factor plots.
4. **`DEEPSPOT_METRIC` auto-pick (section 12)** -- verify the printed pearson-like
   column name is actually the one you want (mean vs. median pearson).
5. **`RNA_DIVERGENCE_EMBEDDINGS` (section 10)** -- defaults to `TOKEN_CENTRAL`;
   the original notebook this was copied from used `"nucleus"` for this specific
   pair, since that's colon's default embeddings segment.

## Extending this notebook

Every section follows the same shape: a small loader/stats helper (shared,
defined once near the top) + a `build_*_table` function + `show_table(...)`
(raw values, auto-exports LaTeX) or a plotting call. A new comparison is a new
`build_*_table` call with different model/token/dataset arguments -- copy the
closest existing section rather than writing a new loader.
